# AudioGen Batch Synthesis Notebook
Headless batch synthesis for Hindi and Punjabi reel scripts.

In [ ]:
MANIFEST_PATH = "scripts.json"
MODEL_WEIGHTS_DIR = "/kaggle/input/audiogen-weights"
OUTPUT_DIR = "outputs"


In [ ]:
import sys
import os
import re
import json
import logging
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("audiogen_batch")

# Add project roots to sys.path
for candidate in [Path.cwd(), Path.cwd().parent, Path("/kaggle/working"), Path("/kaggle/working/audiogen")]:
    src_dir = candidate / "src"
    if src_dir.exists() and str(src_dir) not in sys.path:
        sys.path.insert(0, str(src_dir))
    if (candidate / "voices").exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))
    if (candidate / "batch").exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

from audiogen.engine import Synthesizer
from audiogen.audio_processor import AudioProcessor
from voices.registry import get_voice_ref, VoiceNotFoundError
from batch.manifest_schema import BatchJob, ReelAudioTask

In [ ]:
manifest_file = Path(MANIFEST_PATH)
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

with open(manifest_file, "r", encoding="utf-8") as f:
    raw_manifest = json.load(f)

if isinstance(raw_manifest, list):
    tasks_data = raw_manifest
elif isinstance(raw_manifest, dict) and "tasks" in raw_manifest:
    tasks_data = raw_manifest["tasks"]
else:
    raise ValueError(f"Invalid manifest structure in {manifest_file}")

# Load tasks supporting both pre-validated ReelAudioTask and raw dictionaries
tasks_to_process = []
try:
    if isinstance(raw_manifest, list):
        batch_job = BatchJob(tasks=raw_manifest)
    else:
        batch_job = BatchJob(**raw_manifest)
    tasks_to_process = batch_job.tasks
except Exception:
    tasks_to_process = tasks_data

logger.info(f"Loaded {len(tasks_to_process)} task(s) for batch synthesis.")

# Model loaded ONCE (warm process)
weights_path = Path(MODEL_WEIGHTS_DIR) if MODEL_WEIGHTS_DIR else None
if weights_path is not None and not weights_path.exists():
    logger.warning(f"Weights path {weights_path} not found; falling back to environment/local.")
    weights_path = None

if weights_path and weights_path.exists():
    synthesizer = Synthesizer(model_path=weights_path)
else:
    try:
        synthesizer = Synthesizer()
    except Exception:
        synthesizer = Synthesizer(model_path=Path.cwd())

processor = AudioProcessor(target_sample_rate=24000)
logger.info(f"Initialized Synthesizer on device: {synthesizer.device}")

In [ ]:
results = []
for task_idx, task in enumerate(tasks_to_process):
    if isinstance(task, dict):
        raw_id = str(task.get("id", f"task_{task_idx}")).strip()
        # Sanitize task_id against directory traversal
        safe_id = Path(raw_id).name
        safe_id = re.sub(r"[^a-zA-Z0-9_-]", "_", safe_id).strip("_")
        task_id = safe_id if safe_id else f"task_{task_idx}"
        task_text = task.get("text", "")
        task_lang = task.get("language")
        task_voice = task.get("voice_ref")
    else:
        task_id = getattr(task, "id", f"task_{task_idx}")
        task_text = getattr(task, "text", "")
        task_lang = getattr(task, "language", None)
        task_voice = getattr(task, "voice_ref", None)

    # Ensure out_wav_path is guaranteed strictly inside output_dir
    out_wav_path = (output_dir / f"{task_id}.wav").resolve()
    if not str(out_wav_path).startswith(str(output_dir.resolve())):
        err_msg = f"Security error: task ID '{task_id}' resolves outside output directory"
        logger.error(f"Task '{task_id}' failed: {err_msg}")
        results.append({
            "id": task_id,
            "status": "failed",
            "output_file": None,
            "error": err_msg,
        })
        continue

    logger.info(f"Processing task '{task_id}' (language={task_lang}, voice_ref={task_voice})")

    # 1. Validate language code
    if not task_lang or str(task_lang).strip().lower() not in ("hi", "pa"):
        err_msg = f"Unsupported language '{task_lang}'. Supported languages: ['hi', 'pa']"
        logger.error(f"Task '{task_id}' failed: {err_msg}")
        results.append({
            "id": task_id,
            "status": "failed",
            "output_file": None,
            "error": err_msg,
        })
        continue

    lang = str(task_lang).strip().lower()

    # 2. Resolve voice reference via voice registry
    try:
        voice_rec = get_voice_ref(task_voice)
    except (VoiceNotFoundError, KeyError) as exc:
        err_msg = f"Voice resolution error: Voice '{task_voice}' not found in registry: {exc}"
        logger.error(f"Task '{task_id}' failed: {err_msg}")
        results.append({
            "id": task_id,
            "status": "failed",
            "output_file": None,
            "error": err_msg,
        })
        continue
    except FileNotFoundError as exc:
        err_msg = f"Server configuration error: Reference audio file for voice '{task_voice}' not found on disk: {exc}"
        logger.error(f"Task '{task_id}' failed: {err_msg}")
        results.append({
            "id": task_id,
            "status": "failed",
            "output_file": None,
            "error": err_msg,
        })
        continue
    except Exception as exc:
        err_msg = f"Voice resolution error: {exc}"
        logger.error(f"Task '{task_id}' failed: {err_msg}", exc_info=True)
        results.append({
            "id": task_id,
            "status": "failed",
            "output_file": None,
            "error": err_msg,
        })
        continue

    # 3. Validate language compatibility with resolved voice
    supported_langs = [str(item).strip().lower() for item in voice_rec.language]
    if lang not in supported_langs:
        err_msg = f"Voice '{task_voice}' does not support language '{lang}'. Supported languages: {voice_rec.language}"
        logger.error(f"Task '{task_id}' failed: {err_msg}")
        results.append({
            "id": task_id,
            "status": "failed",
            "output_file": None,
            "error": err_msg,
        })
        continue

    # 4. Invoke Synthesizer.synthesize() and AudioProcessor.process_and_export()
    try:
        raw_audio, sr = synthesizer.synthesize(
            task_text,
            ref_audio_path=voice_rec.path,
            ref_text=voice_rec.ref_text,
        )
        processor.process_and_export(raw_audio, sr, out_wav_path)
        results.append({
            "id": task_id,
            "status": "success",
            "output_file": str(out_wav_path),
            "error": None,
        })
        logger.info(f"Task '{task_id}' completed successfully.")
    except FileNotFoundError as exc:
        err_msg = f"Server configuration error: Reference audio file not found during synthesis: {exc}"
        logger.error(f"Task '{task_id}' failed: {err_msg}")
        results.append({
            "id": task_id,
            "status": "failed",
            "output_file": None,
            "error": err_msg,
        })
    except Exception as exc:
        err_msg = f"Inference failure: {exc}"
        logger.error(f"Task '{task_id}' failed: {err_msg}", exc_info=True)
        results.append({
            "id": task_id,
            "status": "failed",
            "output_file": None,
            "error": err_msg,
        })

manifest_output_path = output_dir / "manifest_output.json"
summary_data = {
    "total": len(results),
    "successful": sum(1 for r in results if r["status"] == "success"),
    "failed": sum(1 for r in results if r["status"] == "failed"),
    "tasks": results,
}
with open(manifest_output_path, "w", encoding="utf-8") as f:
    json.dump(summary_data, f, indent=2)

logger.info(f"Manifest output written to {manifest_output_path}")

In [ ]:
sys.exit(0)
